# LeetCode 1886 — Determine Whether Matrix Can Be Obtained By Rotation bằng Python

Link đề: https://leetcode.com/problems/determine-whether-matrix-can-be-obtained-by-rotation/

Độ khó: **Easy** · Chủ đề: Array, Matrix

```python
class Solution:
    def findRotation(self, mat: list[list[int]], target: list[list[int]]) -> bool:

```

**Mục tiêu học:**

1. Làm quen **ma trận** trong Python: list lồng list, chỉ số `mat[i][j]`, duyệt hai vòng lặp.
2. Tự suy ra công thức xoay 90° theo chiều kim đồng hồ: `new[i][j] = old[n-1-j][i]`.
3. Biết 4 cách giải: tạo ma trận xoay mới · `zip(*mat[::-1])` · xoay tại chỗ (chuyển vị + đảo hàng) · so sánh 4 phép xoay **không cần xoay thật**.
4. Trace vị trí từng ô, chứng minh vì sao chỉ cần thử tối đa 4 góc, phân tích Big-O.
5. Tránh các bẫy: so sánh `tuple` với `list`, sao chép nông (shallow copy), xoay sai chiều.

## 1. Bài toán yêu cầu gì?

Cho hai ma trận vuông `n x n` chỉ chứa `0` và `1`: `mat` và `target`.

Được phép xoay `mat` một góc **90 độ** (bao nhiêu lần tùy ý). Hỏi có thể làm `mat` bằng `target` không?

| mat | target | Kết quả | Vì sao |
|---|---|:---:|---|
| `[[0,1],[1,0]]` | `[[1,0],[0,1]]` | `True` | xoay 90° một lần |
| `[[0,1],[1,1]]` | `[[1,0],[0,1]]` | `False` | `mat` có ba số 1, `target` chỉ có hai |
| `[[0,0,0],[0,1,0],[1,1,1]]` | `[[1,1,1],[0,1,0],[0,0,0]]` | `True` | xoay 2 lần (180°) |

**Ràng buộc:** `1 <= n <= 10`, các phần tử là `0` hoặc `1`.

Quan sát: xoay 4 lần (360°) thì trở về như cũ. Vậy chỉ có **4 trạng thái** khả dĩ: 0°, 90°, 180°, 270°.

In [1]:
examples = [
    ([[0, 1], [1, 0]], [[1, 0], [0, 1]], True),
    ([[0, 1], [1, 1]], [[1, 0], [0, 1]], False),
    ([[0, 0, 0], [0, 1, 0], [1, 1, 1]], [[1, 1, 1], [0, 1, 0], [0, 0, 0]], True),
]

def show(matrix, title=""):
    if title:
        print(title)
    for row in matrix:
        print("  ", row)

for mat, target, expected in examples:
    show(mat, "mat:")
    show(target, "target:")
    print("kỳ vọng:", expected, "\n")

mat:
   [0, 1]
   [1, 0]
target:
   [1, 0]
   [0, 1]
kỳ vọng: True 

mat:
   [0, 1]
   [1, 1]
target:
   [1, 0]
   [0, 1]
kỳ vọng: False 

mat:
   [0, 0, 0]
   [0, 1, 0]
   [1, 1, 1]
target:
   [1, 1, 1]
   [0, 1, 0]
   [0, 0, 0]
kỳ vọng: True 



## 2. Nền tảng Python cần biết

### 2.1. Ma trận = list lồng list

- `mat[i]` là **hàng** thứ `i` (một list).
- `mat[i][j]` là ô ở hàng `i`, cột `j`. Cả hai chỉ số bắt đầu từ 0.
- `n = len(mat)` là số hàng; với ma trận vuông, số cột cũng là `n`.

In [2]:
mat = [[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]]
n = len(mat)
print("n =", n)
print("hàng 1:", mat[1])
print("ô (1, 2):", mat[1][2])      # hàng 1, cột 2 -> 6
print("ô góc dưới trái:", mat[n - 1][0])

n = 3
hàng 1: [4, 5, 6]
ô (1, 2): 6
ô góc dưới trái: 7


### 2.2. Duyệt hai vòng lặp `for i ... for j ...`

Vòng ngoài chọn hàng, vòng trong chọn cột. Thứ tự đọc: trái → phải, trên → dưới.

In [3]:
for i in range(n):
    for j in range(n):
        print(f"({i},{j})={mat[i][j]}", end="  ")
    print()

(0,0)=1  (0,1)=2  (0,2)=3  
(1,0)=4  (1,1)=5  (1,2)=6  
(2,0)=7  (2,1)=8  (2,2)=9  


### 2.3. Tạo ma trận rỗng bằng list comprehension — và cái bẫy `*`

Muốn ma trận `n x n` toàn số 0:

- **Đúng:** `[[0] * n for _ in range(n)]` — mỗi hàng là một list **riêng**.
- **Sai:** `[[0] * n] * n` — cả `n` hàng là **cùng một list**; sửa một ô thì cả cột đổi theo.

In [4]:
good = [[0] * 3 for _ in range(3)]
bad = [[0] * 3] * 3
good[0][0] = 1
bad[0][0] = 1
print("good:", good)
print("bad :", bad, " <- cả 3 hàng đều đổi!")
print("bad[0] is bad[1]:", bad[0] is bad[1])

good: [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
bad : [[1, 0, 0], [1, 0, 0], [1, 0, 0]]  <- cả 3 hàng đều đổi!
bad[0] is bad[1]: True


### 2.4. So sánh hai ma trận bằng `==`

Hai list bằng nhau khi cùng độ dài và từng phần tử bằng nhau — áp dụng đệ quy cho list lồng nhau. Nhưng **`list` không bằng `tuple`** dù cùng nội dung.

In [5]:
print([[1, 0], [0, 1]] == [[1, 0], [0, 1]])   # True
print([[1, 0], [0, 1]] == [[1, 0], [1, 1]])   # False
print([1, 0] == (1, 0))                       # False! list khác tuple
print([1, 0] == list((1, 0)))                 # True

True
False
False
True


### 2.5. `zip(*rows)` và `rows[::-1]`

- `rows[::-1]` đảo thứ tự các **hàng** (hàng cuối lên đầu).
- `zip(a, b, c)` ghép phần tử cùng vị trí: `(a[0], b[0], c[0])`, `(a[1], b[1], c[1])`, ...
- `zip(*mat)` "trải" các hàng làm đối số → mỗi tuple sinh ra là một **cột** của `mat`. Đó là phép **chuyển vị** (transpose).

In [6]:
print("đảo hàng :", mat[::-1])
print("chuyển vị:", list(zip(*mat)))
print("đảo hàng rồi chuyển vị:", [list(col) for col in zip(*mat[::-1])])

đảo hàng : [[7, 8, 9], [4, 5, 6], [1, 2, 3]]
chuyển vị: [(1, 4, 7), (2, 5, 8), (3, 6, 9)]
đảo hàng rồi chuyển vị: [[7, 4, 1], [8, 5, 2], [9, 6, 3]]


## 3. Tính bằng tay: ô nào đi đâu khi xoay 90° theo chiều kim đồng hồ?

Xoay ma trận `3 x 3` chứa số 1..9:

```text
1 2 3        7 4 1
4 5 6   →    8 5 2
7 8 9        9 6 3
```

Nhìn **hàng 0 của kết quả** `7 4 1`: đó là **cột 0** của ma trận cũ, đọc từ **dưới lên**.

Tổng quát: hàng `i` của ma trận mới = cột `i` của ma trận cũ, đọc từ dưới lên. Ô `new[i][j]` lấy từ cột `i`, hàng thứ `j` tính từ dưới, tức hàng `n-1-j`:

```python
new[i][j] = old[n - 1 - j][i]
```

Kiểm tra: `new[0][0] = old[2][0] = 7` ✓, `new[0][2] = old[0][0] = 1` ✓, `new[2][1] = old[1][2] = 6` ✓.

Cách nhìn ngược lại (ô cũ đi đâu): `old[i][j]` → `new[j][n-1-i]`.

In [7]:
def rotate_by_formula(old):
    n = len(old)
    new = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            new[i][j] = old[n - 1 - j][i]
    return new

show(rotate_by_formula(mat), "xoay 90° lần 1:")
show(rotate_by_formula(rotate_by_formula(mat)), "xoay 90° lần 2 (180°):")

xoay 90° lần 1:
   [7, 4, 1]
   [8, 5, 2]
   [9, 6, 3]
xoay 90° lần 2 (180°):
   [9, 8, 7]
   [6, 5, 4]
   [3, 2, 1]


## 4. Cách 1 — Tạo ma trận xoay mới, thử tối đa 4 lần

Ý tưởng tự nhiên:

1. Nếu `mat == target` → `True` (xoay 0°).
2. Xoay `mat` 90°, so sánh lại. Lặp tối đa 4 lần.
3. Hết 4 lần mà không khớp → `False`.

Vì sao **4** là đủ? Xoay 4 lần trở lại ban đầu, nên lần thứ 5 trở đi chỉ lặp lại các trạng thái cũ.

In [8]:
def find_rotation_new_matrix(mat, target):
    cur = mat
    for k in range(4):
        if cur == target:
            return True
        cur = rotate_by_formula(cur)
    return False

for mat_, target_, expected in examples:
    print(find_rotation_new_matrix(mat_, target_), "| kỳ vọng", expected)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


In [9]:
# Xác nhận: xoay 4 lần trở về ban đầu
cur = mat
for _ in range(4):
    cur = rotate_by_formula(cur)
print(cur == mat)

True


### Code nộp LeetCode — Cách 1

In [10]:
class Solution:
    def findRotation(self, mat: list[list[int]], target: list[list[int]]) -> bool:
        n = len(mat)
        for _ in range(4):
            if mat == target:
                return True
            mat = [[mat[n - 1 - j][i] for j in range(n)] for i in range(n)]
        return False


print(Solution().findRotation([[0, 1], [1, 0]], [[1, 0], [0, 1]]))  # True

True


**Độ phức tạp:** mỗi lần xoay và mỗi lần so sánh tốn `O(n²)`; có tối đa 4 lần → `O(4n²) = O(n²)` thời gian. Bộ nhớ `O(n²)` cho ma trận mới.

List comprehension lồng nhau `[[... for j in range(n)] for i in range(n)]` đọc như sau: vòng **ngoài cùng bên phải** (`for i`) tạo từng hàng; vòng bên trong (`for j`) tạo từng ô của hàng đó.

## 5. Cách 2 — Xoay bằng một dòng: `zip(*mat[::-1])`

Ở mục 3 ta thấy: hàng `i` mới = cột `i` cũ đọc từ dưới lên. Vậy:

1. `mat[::-1]` — đảo hàng (để "đọc từ dưới lên").
2. `zip(*...)` — lấy cột thành hàng.

Nhớ đổi mỗi tuple thành list, nếu không phép `==` với `target` sẽ luôn `False`.

In [11]:
def rotate_zip(m):
    return [list(row) for row in zip(*m[::-1])]

print(rotate_zip(mat) == rotate_by_formula(mat))

def find_rotation_zip(mat, target):
    for _ in range(4):
        if mat == target:
            return True
        mat = rotate_zip(mat)
    return False

for mat_, target_, expected in examples:
    print(find_rotation_zip(mat_, target_), "| kỳ vọng", expected)

True
True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


### Code nộp LeetCode — Cách 2

In [12]:
class Solution:
    def findRotation(self, mat: list[list[int]], target: list[list[int]]) -> bool:
        for _ in range(4):
            if mat == target:
                return True
            mat = [list(row) for row in zip(*mat[::-1])]
        return False


print(Solution().findRotation([[0, 1], [1, 1]], [[1, 0], [0, 1]]))  # False

False


## 6. Cách 3 — Xoay **tại chỗ**: chuyển vị rồi đảo từng hàng

Đây là kỹ thuật của LeetCode 48 (Rotate Image). Xoay 90° kim đồng hồ = **chuyển vị** (đổi `m[i][j]` với `m[j][i]`) rồi **đảo mỗi hàng**.

```text
1 2 3   chuyển vị   1 4 7   đảo hàng   7 4 1
4 5 6   ────────►   2 5 8   ───────►   8 5 2
7 8 9               3 6 9              9 6 3
```

Chỉ swap trong tam giác trên (`j > i`) để mỗi cặp đổi đúng một lần. Swap trong Python: `a, b = b, a`.

Lưu ý: cách này **sửa trực tiếp** `mat` của người gọi. Trong notebook ta sao chép trước bằng `[row[:] for row in mat]` để không phá dữ liệu test.

In [13]:
def rotate_in_place(m):
    n = len(m)
    for i in range(n):
        for j in range(i + 1, n):
            m[i][j], m[j][i] = m[j][i], m[i][j]
    for row in m:
        row.reverse()

demo = [row[:] for row in mat]     # bản sao sâu một tầng
rotate_in_place(demo)
show(demo, "xoay tại chỗ:")
print("mat gốc không đổi:", mat)

xoay tại chỗ:
   [7, 4, 1]
   [8, 5, 2]
   [9, 6, 3]
mat gốc không đổi: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]


In [14]:
def find_rotation_in_place(mat, target):
    m = [row[:] for row in mat]
    for _ in range(4):
        if m == target:
            return True
        rotate_in_place(m)
    return False

for mat_, target_, expected in examples:
    print(find_rotation_in_place(mat_, target_), "| kỳ vọng", expected)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


**Độ phức tạp:** thời gian `O(n²)`; bộ nhớ phụ `O(1)` nếu được phép sửa `mat` (trên LeetCode thì được). Trong notebook ta có thêm một bản sao `O(n²)` chỉ để bảo vệ dữ liệu.

## 7. Insight — so sánh 4 góc xoay **mà không cần xoay thật**

Nút thắt của 3 cách trên: ta **dựng lại** ma trận rồi mới so sánh. Nhưng thật ra chỉ cần biết ô `mat[i][j]` sẽ nằm ở đâu trong mỗi phép xoay:

| Góc | `mat[i][j]` đi tới ô | Điều kiện khớp |
|---|---|---|
| 0° | `(i, j)` | `mat[i][j] == target[i][j]` |
| 90° | `(j, n-1-i)` | `mat[i][j] == target[j][n-1-i]` |
| 180° | `(n-1-i, n-1-j)` | `mat[i][j] == target[n-1-i][n-1-j]` |
| 270° | `(n-1-j, i)` | `mat[i][j] == target[n-1-j][i]` |

Giữ 4 cờ `ok[0..3]`, ban đầu `True`. Duyệt mọi ô một lần; ô nào không khớp góc `k` thì tắt cờ `k`. Cuối cùng trả `any(ok)`.

→ Một lượt duyệt, **không** cấp phát ma trận mới, không sửa input.

In [15]:
# Kiểm tra công thức từng góc bằng cách so với rotate_by_formula
n3 = len(mat)
rot = mat
positions = {
    0: lambda i, j: (i, j),
    1: lambda i, j: (j, n3 - 1 - i),
    2: lambda i, j: (n3 - 1 - i, n3 - 1 - j),
    3: lambda i, j: (n3 - 1 - j, i),
}
for k in range(4):
    good_k = all(rot[r][cc] == mat[i][j]
                 for i in range(n3) for j in range(n3)
                 for (r, cc) in [positions[k](i, j)])
    print(f"góc {90 * k:3}°: công thức vị trí đúng = {good_k}")
    rot = rotate_by_formula(rot)

góc   0°: công thức vị trí đúng = True
góc  90°: công thức vị trí đúng = True
góc 180°: công thức vị trí đúng = True
góc 270°: công thức vị trí đúng = True


`all(...)` trả `True` nếu **mọi** phần tử đều đúng; `any(...)` trả `True` nếu **có ít nhất một** phần tử đúng. Cả hai dừng sớm khi đã biết kết quả.

In [16]:
print(all([True, True, False]), any([True, True, False]))
print(all([]), any([]))   # quy ước: all rỗng = True, any rỗng = False

False True
True False


In [17]:
def find_rotation_flags(mat, target):
    n = len(mat)
    ok = [True, True, True, True]
    for i in range(n):
        for j in range(n):
            v = mat[i][j]
            if v != target[i][j]:
                ok[0] = False
            if v != target[j][n - 1 - i]:
                ok[1] = False
            if v != target[n - 1 - i][n - 1 - j]:
                ok[2] = False
            if v != target[n - 1 - j][i]:
                ok[3] = False
    return any(ok)

for mat_, target_, expected in examples:
    print(find_rotation_flags(mat_, target_), "| kỳ vọng", expected)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


## 8. TRACE từng bước (Cách 4 — bốn cờ)

Ví dụ 1: `mat = [[0,1],[1,0]]`, `target = [[1,0],[0,1]]`, `n = 2`.

| (i,j) | v | 0°: target[i][j] | 90°: target[j][1-i] | 180°: target[1-i][1-j] | 270°: target[1-j][i] | ok sau bước |
|---|:-:|:-:|:-:|:-:|:-:|---|
| (0,0) | 0 | 1 ✗ | target[0][1]=0 ✓ | target[1][1]=1 ✗ | target[1][0]=0 ✓ | `[F,T,F,T]` |
| (0,1) | 1 | 0 ✗ | target[1][1]=1 ✓ | target[1][0]=0 ✗ | target[0][0]=1 ✓ | `[F,T,F,T]` |
| (1,0) | 1 | 0 ✗ | target[0][0]=1 ✓ | target[0][1]=0 ✗ | target[1][1]=1 ✓ | `[F,T,F,T]` |
| (1,1) | 0 | 1 ✗ | target[1][0]=0 ✓ | target[0][0]=1 ✗ | target[0][1]=0 ✓ | `[F,T,F,T]` |

Kết quả `any([F,T,F,T]) = True`. Thú vị: với ma trận này, cả 90° **và** 270° đều khớp (vì xoay 180° giữ nguyên `mat`).

In [18]:
def trace_find_rotation(mat, target):
    n = len(mat)
    ok = [True] * 4
    names = ["0°", "90°", "180°", "270°"]
    for i in range(n):
        for j in range(n):
            v = mat[i][j]
            cells = [(i, j), (j, n - 1 - i), (n - 1 - i, n - 1 - j), (n - 1 - j, i)]
            marks = []
            for k, (r, cc) in enumerate(cells):
                match = target[r][cc] == v
                if not match:
                    ok[k] = False
                marks.append(f"{names[k]}:t[{r}][{cc}]={target[r][cc]}{'✓' if match else '✗'}")
            print(f"({i},{j}) v={v} | " + "  ".join(marks) + f" | ok={ok}")
    print("Kết quả any(ok) =", any(ok))
    return any(ok)

trace_find_rotation([[0, 1], [1, 0]], [[1, 0], [0, 1]])

(0,0) v=0 | 0°:t[0][0]=1✗  90°:t[0][1]=0✓  180°:t[1][1]=1✗  270°:t[1][0]=0✓ | ok=[False, True, False, True]
(0,1) v=1 | 0°:t[0][1]=0✗  90°:t[1][1]=1✓  180°:t[1][0]=0✗  270°:t[0][0]=1✓ | ok=[False, True, False, True]
(1,0) v=1 | 0°:t[1][0]=0✗  90°:t[0][0]=1✓  180°:t[0][1]=0✗  270°:t[1][1]=1✓ | ok=[False, True, False, True]
(1,1) v=0 | 0°:t[1][1]=1✗  90°:t[1][0]=0✓  180°:t[0][0]=1✗  270°:t[0][1]=0✓ | ok=[False, True, False, True]
Kết quả any(ok) = True


True

### Trace ví dụ `False` — thấy cả 4 cờ lần lượt bị tắt

In [19]:
trace_find_rotation([[0, 1], [1, 1]], [[1, 0], [0, 1]])

(0,0) v=0 | 0°:t[0][0]=1✗  90°:t[0][1]=0✓  180°:t[1][1]=1✗  270°:t[1][0]=0✓ | ok=[False, True, False, True]
(0,1) v=1 | 0°:t[0][1]=0✗  90°:t[1][1]=1✓  180°:t[1][0]=0✗  270°:t[0][0]=1✓ | ok=[False, True, False, True]
(1,0) v=1 | 0°:t[1][0]=0✗  90°:t[0][0]=1✓  180°:t[0][1]=0✗  270°:t[1][1]=1✓ | ok=[False, True, False, True]
(1,1) v=1 | 0°:t[1][1]=1✓  90°:t[1][0]=0✗  180°:t[0][0]=1✓  270°:t[0][1]=0✗ | ok=[False, False, False, False]
Kết quả any(ok) = False


False

### Trace ví dụ 3 bằng Cách 1 — xem ma trận sau từng lần xoay

In [20]:
def trace_rotations(mat, target):
    cur = mat
    for k in range(4):
        print(f"Sau {k} lần xoay ({90 * k}°): {cur} -> khớp target? {cur == target}")
        if cur == target:
            return True
        cur = rotate_by_formula(cur)
    return False

trace_rotations([[0, 0, 0], [0, 1, 0], [1, 1, 1]], [[1, 1, 1], [0, 1, 0], [0, 0, 0]])

Sau 0 lần xoay (0°): [[0, 0, 0], [0, 1, 0], [1, 1, 1]] -> khớp target? False
Sau 1 lần xoay (90°): [[1, 0, 0], [1, 1, 0], [1, 0, 0]] -> khớp target? False
Sau 2 lần xoay (180°): [[1, 1, 1], [0, 1, 0], [0, 0, 0]] -> khớp target? True


True

## 9. Code nộp LeetCode — lời giải chính (bốn cờ, không xoay)

Có thể dừng sớm khi cả 4 cờ đều đã `False`.

In [21]:
class Solution:
    def findRotation(self, mat: list[list[int]], target: list[list[int]]) -> bool:
        n = len(mat)
        ok = [True] * 4
        for i in range(n):
            for j in range(n):
                v = mat[i][j]
                if v != target[i][j]:
                    ok[0] = False
                if v != target[j][n - 1 - i]:
                    ok[1] = False
                if v != target[n - 1 - i][n - 1 - j]:
                    ok[2] = False
                if v != target[n - 1 - j][i]:
                    ok[3] = False
            if not any(ok):
                return False
        return any(ok)


sol = Solution()
for mat_, target_, expected in examples:
    assert sol.findRotation(mat_, target_) == expected
print("Solution (bốn cờ) PASS 3 ví dụ")

Solution (bốn cờ) PASS 3 ví dụ


## 10. Vì sao đúng?

**Chỉ cần 4 góc.** Gọi `R` là phép xoay 90°. Ta có `R⁴ = I` (xoay 360° là giữ nguyên). Mọi số lần xoay `k` cho cùng kết quả với `k mod 4`. Nên tập trạng thái đạt được là `{mat, R(mat), R²(mat), R³(mat)}` — đúng 4 trạng thái mà mọi cách ở trên đều kiểm tra.

**Công thức xoay đúng.** Hàng `i` của ảnh xoay là cột `i` của ảnh gốc đọc từ dưới lên → `new[i][j] = old[n-1-j][i]`. Đặt `r = n-1-j, c = i` thì ô gốc `(r, c)` đi tới `(c, n-1-r)` — trùng với dòng 90° trong bảng mục 7. Áp dụng 2, 3 lần ra các dòng 180°, 270°.

**Bốn cờ đúng.** Invariant: sau khi duyệt một tập ô `S`, `ok[k] == True` khi và chỉ khi mọi ô trong `S` khớp giữa `R^k(mat)` và `target`. Khi `S` là toàn bộ ma trận, `ok[k]` đúng nghĩa là `R^k(mat) == target`. Phép tương ứng ô `(i,j)` → vị trí mới là **song ánh** (mỗi ô đích nhận đúng một ô nguồn), nên duyệt hết ô nguồn là đã kiểm tra hết ô đích.

## 11. Độ phức tạp và so sánh các cách

`n` = cạnh ma trận (`n ≤ 10`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Sửa input? | Ghi chú |
|---|---|:---:|:---:|:---:|---|
| 1. Ma trận mới theo công thức | `new[i][j]=old[n-1-j][i]` | `O(n²)` | `O(n²)` | Không | dễ hiểu, dễ kiểm |
| 2. `zip(*mat[::-1])` | đảo hàng + chuyển vị | `O(n²)` | `O(n²)` | Không | ngắn nhất, nhớ đổi tuple→list |
| 3. Xoay tại chỗ | chuyển vị + đảo hàng | `O(n²)` | `O(1)` | Có | kỹ thuật của LeetCode 48 |
| 4. Bốn cờ | so vị trí, không xoay | `O(n²)` | `O(1)` | Không | một lượt duyệt, dừng sớm |

Cả bốn đều `O(n²)` vì phải nhìn mọi ô ít nhất một lần. Hằng số: cách 1–3 tối đa 4 lần xoay + 4 lần so sánh; cách 4 đúng một lượt (mỗi ô 4 phép so sánh).

## 12. Bộ test kiểm tra

Nhóm test: 3 ví dụ của đề; `n = 1` (khớp / không khớp); ma trận cần đúng 270°; ma trận đối xứng (mọi góc đều khớp); khác số lượng số 1; đối chiếu 4 cách trên ma trận ngẫu nhiên (có seed) — target sinh bằng cách xoay ngẫu nhiên hoặc lật một bit.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:36} PASS")

solvers = [find_rotation_new_matrix, find_rotation_zip, find_rotation_in_place, find_rotation_flags]

m270 = [[1, 2], [3, 4]]
t270 = rotate_by_formula(rotate_by_formula(rotate_by_formula(m270)))
tests = [
    ("ví dụ 1", examples[0][0], examples[0][1], True),
    ("ví dụ 2", examples[1][0], examples[1][1], False),
    ("ví dụ 3", examples[2][0], examples[2][1], True),
    ("n=1 khớp", [[1]], [[1]], True),
    ("n=1 không khớp", [[0]], [[1]], False),
    ("cần đúng 270°", m270, t270, True),
    ("toàn 0", [[0, 0], [0, 0]], [[0, 0], [0, 0]], True),
    ("lật gương (không phải xoay)", [[1, 0], [0, 0]], [[0, 1], [0, 0]], True),
    ("phản xạ chéo không đạt", [[1, 1, 0], [0, 0, 0], [0, 0, 0]], [[1, 0, 0], [1, 0, 0], [0, 0, 0]], False),
]
for name, a, t, expected in tests:
    for f in solvers:
        check(f"{name} [{f.__name__[14:]}]", f(a, t), expected)

ví dụ 1 [new_matrix]                 PASS
ví dụ 1 [zip]                        PASS
ví dụ 1 [in_place]                   PASS
ví dụ 1 [flags]                      PASS
ví dụ 2 [new_matrix]                 PASS
ví dụ 2 [zip]                        PASS
ví dụ 2 [in_place]                   PASS
ví dụ 2 [flags]                      PASS
ví dụ 3 [new_matrix]                 PASS
ví dụ 3 [zip]                        PASS
ví dụ 3 [in_place]                   PASS
ví dụ 3 [flags]                      PASS
n=1 khớp [new_matrix]                PASS
n=1 khớp [zip]                       PASS
n=1 khớp [in_place]                  PASS
n=1 khớp [flags]                     PASS
n=1 không khớp [new_matrix]          PASS
n=1 không khớp [zip]                 PASS
n=1 không khớp [in_place]            PASS
n=1 không khớp [flags]               PASS
cần đúng 270° [new_matrix]           PASS
cần đúng 270° [zip]                  PASS
cần đúng 270° [in_place]             PASS
cần đúng 270° [flags]             

Ghi chú ca "lật gương": `[[1,0],[0,0]]` lật trái-phải thành `[[0,1],[0,0]]`, và ở `2x2` điều đó **trùng** với xoay 90°, nên `True`. Ca cuối: `[[1,1,0],...]` phản xạ qua đường chéo chính thành `[[1,0,0],[1,0,0],...]`; xoay 270° của `mat` là `[[0,0,0],[1,0,0],[1,0,0]]` ≠ target → `False`. Hãy tự xoay tay để thấy phản xạ ≠ xoay.

In [23]:
import random

random.seed(1886)
mismatch = 0
for _ in range(1500):
    n_ = random.randint(1, 5)
    a = [[random.randint(0, 1) for _ in range(n_)] for _ in range(n_)]
    t = a
    for _ in range(random.randint(0, 3)):
        t = rotate_by_formula(t)
    if random.random() < 0.5:
        t = [row[:] for row in t]
        r, cc = random.randrange(n_), random.randrange(n_)
        t[r][cc] ^= 1          # lật một bit để tạo nhiều ca False
    results = {f(a, t) for f in solvers}
    if len(results) != 1:
        mismatch += 1
print("Số ca các cách cho kết quả khác nhau:", mismatch)
assert mismatch == 0

Số ca các cách cho kết quả khác nhau: 0


`t[r][cc] ^= 1` dùng phép **XOR** với 1: biến 0 thành 1 và 1 thành 0 — cách nhanh để "lật bit".

## 13. Lỗi người mới hay gặp

### Lỗi 1: So sánh tuple với list

```python
mat = list(zip(*mat[::-1]))   # SAI: mỗi hàng là tuple
return mat == target          # luôn False
```

Sửa: `[list(row) for row in zip(*mat[::-1])]`.

In [24]:
a = [[0, 1], [1, 0]]
t = [[1, 0], [0, 1]]
wrong = list(zip(*a[::-1]))
print("Sai :", wrong, "==", t, "->", wrong == t)
right = [list(row) for row in zip(*a[::-1])]
print("Đúng:", right, "==", t, "->", right == t)

Sai : [(1, 0), (0, 1)] == [[1, 0], [0, 1]] -> False
Đúng: [[1, 0], [0, 1]] == [[1, 0], [0, 1]] -> True


### Lỗi 2: Xoay tại chỗ mà swap cả tam giác dưới

```python
for i in range(n):
    for j in range(n):          # SAI: mỗi cặp bị swap 2 lần → trở lại như cũ
        m[i][j], m[j][i] = m[j][i], m[i][j]
```

Sửa: `for j in range(i + 1, n)`.

In [25]:
def wrong_transpose(m):
    n = len(m)
    for i in range(n):
        for j in range(n):
            m[i][j], m[j][i] = m[j][i], m[i][j]

x = [[1, 2], [3, 4]]
wrong_transpose(x)
print("Sau 'chuyển vị' sai:", x, "(không đổi gì!)")

Sau 'chuyển vị' sai: [[1, 2], [3, 4]] (không đổi gì!)


### Lỗi 3: Ghi vào chính ma trận đang đọc

```python
for i in range(n):
    for j in range(n):
        mat[i][j] = mat[n - 1 - j][i]   # SAI: các ô đã ghi đè bị đọc lại
```

Vì sao sai: khi tới ô sau, giá trị nguồn có thể đã bị ghi đè ở bước trước. Sửa: ghi vào ma trận **mới** (Cách 1), hoặc dùng chuyển vị + đảo hàng (Cách 3).

In [26]:
y = [[1, 2], [3, 4]]
n_y = 2
for i in range(n_y):
    for j in range(n_y):
        y[i][j] = y[n_y - 1 - j][i]
print("Sai :", y)
print("Đúng:", rotate_by_formula([[1, 2], [3, 4]]))

Sai : [[3, 3], [4, 3]]
Đúng: [[3, 1], [4, 2]]


### Lỗi 4: Chỉ thử 1 lần xoay hoặc quên góc 0°

Nhiều bạn chỉ xoay rồi so, quên rằng **không xoay** (0°) cũng hợp lệ: nếu `mat == target` ngay từ đầu thì phải trả `True`. Kiểm tra trước khi xoay, lặp đủ 4 lần.

### Lỗi 5: Sao chép nông `mat.copy()` rồi xoay tại chỗ

`mat.copy()` chỉ sao chép list ngoài; các hàng bên trong vẫn **dùng chung**. Xoay tại chỗ trên bản sao vẫn làm hỏng `mat`. Dùng `[row[:] for row in mat]`.

In [27]:
orig = [[1, 2], [3, 4]]
shallow = orig.copy()
rotate_in_place(shallow)
print("orig sau khi xoay bản sao NÔNG:", orig, "<- bị hỏng")

orig = [[1, 2], [3, 4]]
deep = [row[:] for row in orig]
rotate_in_place(deep)
print("orig sau khi xoay bản sao từng hàng:", orig, "<- an toàn")

orig sau khi xoay bản sao NÔNG: [[3, 1], [4, 2]] <- bị hỏng
orig sau khi xoay bản sao từng hàng: [[1, 2], [3, 4]] <- an toàn


## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:** bài ma trận vuông + "xoay / lật / đối xứng" → nghĩ ngay tới:

- bảng ánh xạ vị trí `(i, j)` → `(j, n-1-i)` ...;
- xoay = chuyển vị + đảo hàng (kim đồng hồ) hoặc chuyển vị + đảo cột / đảo hàng rồi chuyển vị (ngược kim);
- số trạng thái hữu hạn (4 góc) → thử hết.

Bài liên quan: 48 (Rotate Image), 867 (Transpose Matrix), 832 (Flipping an Image), 1260 (Shift 2D Grid).

### Biến thể 1: cho phép cả xoay **và** lật gương

Nhóm đối xứng của hình vuông có 8 phần tử (4 góc xoay × có/không lật). Chỉ cần thử thêm 4 góc của ma trận đã lật.

In [28]:
def find_rotation_or_flip(mat, target):
    flipped = [row[::-1] for row in mat]
    return find_rotation_flags(mat, target) or find_rotation_flags(flipped, target)

print(find_rotation_or_flip([[1, 1, 0], [0, 0, 0], [0, 0, 0]],
                            [[1, 0, 0], [1, 0, 0], [0, 0, 0]]))  # phản xạ chéo = lật + xoay -> True

True


### Biến thể 2: số lần xoay **ít nhất**

Trả về `k ∈ {0,1,2,3}` nhỏ nhất sao cho `R^k(mat) == target`, hoặc `-1`. Từ bốn cờ: tìm chỉ số đầu tiên còn `True`.

In [29]:
def min_rotations(mat, target):
    cur = mat
    for k in range(4):
        if cur == target:
            return k
        cur = rotate_by_formula(cur)
    return -1

print(min_rotations(*examples[2][:2]), min_rotations(*examples[1][:2]))

2 -1


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay

Không chạy code: xoay `[[1,2,3],[4,5,6],[7,8,9]]` **ngược** chiều kim đồng hồ 90°. Ghi kết quả vào `ex1_answer`.

In [30]:
ex1_answer = None   # TODO: ma trận 3x3

# assert ex1_answer == ...  (so với đáp án ở cuối)

### Bài 2 — Điền chỗ trống: xoay 90° theo chiều kim đồng hồ ra ma trận mới

In [31]:
def ex2_rotate(old):
    n = len(old)
    new = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            # TODO: new[i][j] = ?
            pass
    return new

# assert ex2_rotate([[1, 2], [3, 4]]) == [[3, 1], [4, 2]]

### Bài 3 — Xoay ngược chiều kim đồng hồ tại chỗ

Viết `rotate_ccw_in_place(m)` (không tạo ma trận mới). Gợi ý: chuyển vị + một thao tác đảo khác.

In [32]:
def rotate_ccw_in_place(m):
    # TODO
    pass

# x = [[1, 2], [3, 4]]; rotate_ccw_in_place(x); assert x == [[2, 4], [1, 3]]

### Bài 4 — Tự viết lại `findRotation` bằng `zip`, không xem code phía trên

In [33]:
def ex4_find_rotation(mat, target):
    # TODO
    pass

# for a, t, e in examples: assert ex4_find_rotation(a, t) == e

### Bài 5 — Xoay ma trận **chữ nhật** `m x n`

Viết `rotate_rect(mat)` xoay 90° kim đồng hồ ma trận `m x n` → kết quả `n x m`. Ví dụ `[[1,2,3],[4,5,6]]` → `[[4,1],[5,2],[6,3]]`.

In [34]:
def rotate_rect(mat):
    # TODO
    pass

# assert rotate_rect([[1, 2, 3], [4, 5, 6]]) == [[4, 1], [5, 2], [6, 3]]

### Bài 6 — Mini challenge: đếm số góc khớp

Viết `count_matching_rotations(mat, target)` trả về **số** góc trong `{0°, 90°, 180°, 270°}` làm `mat` thành `target` (0..4), chỉ duyệt ma trận **một lần**.

In [35]:
def count_matching_rotations(mat, target):
    # TODO: dùng bốn cờ
    pass

# assert count_matching_rotations([[0, 1], [1, 0]], [[1, 0], [0, 1]]) == 2
# assert count_matching_rotations([[1]], [[1]]) == 4

## 16. Gợi ý 3 mức

**Bài 1.** (1) Ngược chiều kim đồng hồ: hàng đầu mới là **cột cuối** cũ, đọc từ trên xuống. (2) Cột cuối là `3,6,9`. (3) Kết quả bắt đầu bằng `[3,6,9]`.

**Bài 2.** (1) Hàng `i` mới = cột `i` cũ đọc từ dưới lên. (2) Hàng `n-1-j`, cột `i`. (3) `new[i][j] = old[n - 1 - j][i]`.

**Bài 3.** (1) Xoay ngược = chuyển vị rồi đảo **thứ tự hàng** (hoặc đảo từng hàng rồi chuyển vị). (2) Swap tam giác trên; `m.reverse()`. (3) Sau chuyển vị gọi `m.reverse()` (đảo list ngoài).

**Bài 4.** (1) Lặp 4 lần. (2) So trước, xoay sau. (3) `mat = [list(r) for r in zip(*mat[::-1])]`.

**Bài 5.** (1) `zip` hoạt động cả với ma trận chữ nhật. (2) Kết quả có `n` hàng. (3) `[list(r) for r in zip(*mat[::-1])]`.

**Bài 6.** (1) Giống lời giải chính. (2) Không trả `any(ok)` mà đếm. (3) `sum(ok)` — `True` được tính là 1.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [36]:
ans_ex1 = [[3, 6, 9], [2, 5, 8], [1, 4, 7]]


def ans_ex2_rotate(old):
    n = len(old)
    new = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            new[i][j] = old[n - 1 - j][i]
    return new


def ans_rotate_ccw_in_place(m):
    n = len(m)
    for i in range(n):
        for j in range(i + 1, n):
            m[i][j], m[j][i] = m[j][i], m[i][j]
    m.reverse()


def ans_ex4_find_rotation(mat, target):
    for _ in range(4):
        if mat == target:
            return True
        mat = [list(r) for r in zip(*mat[::-1])]
    return False


def ans_rotate_rect(mat):
    return [list(r) for r in zip(*mat[::-1])]


def ans_count_matching_rotations(mat, target):
    n = len(mat)
    ok = [True] * 4
    for i in range(n):
        for j in range(n):
            v = mat[i][j]
            ok[0] &= v == target[i][j]
            ok[1] &= v == target[j][n - 1 - i]
            ok[2] &= v == target[n - 1 - i][n - 1 - j]
            ok[3] &= v == target[n - 1 - j][i]
    return sum(ok)


# Kiểm tra đáp án
cw3 = rotate_by_formula(rotate_by_formula(rotate_by_formula([[1, 2, 3], [4, 5, 6], [7, 8, 9]])))
assert ans_ex1 == cw3                       # ngược 90° = thuận 270°
assert ans_ex2_rotate([[1, 2], [3, 4]]) == [[3, 1], [4, 2]]
x = [[1, 2], [3, 4]]; ans_rotate_ccw_in_place(x); assert x == [[2, 4], [1, 3]]
for a, t, e in examples:
    assert ans_ex4_find_rotation(a, t) == e
assert ans_rotate_rect([[1, 2, 3], [4, 5, 6]]) == [[4, 1], [5, 2], [6, 3]]
assert ans_count_matching_rotations([[0, 1], [1, 0]], [[1, 0], [0, 1]]) == 2
assert ans_count_matching_rotations([[1]], [[1]]) == 4
assert ans_count_matching_rotations(*examples[1][:2]) == 0
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


`ok[0] &= cond` nghĩa là `ok[0] = ok[0] and cond` (với giá trị bool) — một khi đã `False` thì giữ `False`.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findRotation(self, mat, target):
        for _ in range(4):
            if mat == target:
                return True
            mat = [list(row) for row in zip(*mat[::-1])]   # xoay 90° kim đồng hồ
        return False
```

**Công thức cần nhớ:**

- Xoay thuận: `new[i][j] = old[n-1-j][i]` · `zip(*m[::-1])` · chuyển vị + đảo từng hàng.
- Xoay ngược: `new[i][j] = old[j][n-1-i]` · chuyển vị + đảo thứ tự hàng.

**Checklist trước khi submit:**

- [ ] Có kiểm tra góc 0° (trước khi xoay)?
- [ ] Lặp đủ 4 góc, không hơn?
- [ ] Đổi tuple → list nếu dùng `zip`?
- [ ] Không ghi đè ô đang cần đọc?
- [ ] Tên method `findRotation`, trả về `bool`?